# LC #3 — Longest Substring Without Repeating Characters
**Category:** Sliding Window + HashSet | **Difficulty:** Medium | **Day 2**

---

<div style="padding:15px;border-left:8px solid #11998e;background:#e8faf5;border-radius:4px;">
<strong>The Problem:</strong> Given string <code>s</code>, return the length of the longest substring
without repeating characters.
</div>

**Examples:**
```
"abcabcbb" → 3   ("abc")
"bbbbb"    → 1   ("b")
"pwwkew"   → 3   ("wke")
""         → 0
```

**Core Insight:** Two pointers. Right expands to grow the window. When a repeat is found, jump left pointer past the previous occurrence of that character — not just one step, but directly to `prev_index + 1`.

## Mental Models

**1. The Valid Window**
A window `[left, right]` is valid if all characters within it are unique. Expanding right may invalidate it. Shrinking left restores validity.

**2. Jump Left, Don't Shuffle**
When we see a repeat at position `right`, we don't shrink left one step at a time. We jump left to `char_index[char] + 1` — past the previous occurrence. This is what makes it O(n) rather than O(n²).

**3. Why Store Index (Not Just Membership)?**
If we only tracked whether a char is in the window (set membership), we'd have to shrink one step at a time. Storing the index lets us jump directly.

In [ ]:
# Brute Force — O(n³) time, O(min(n,m)) space
# Check every substring for uniqueness.

def lengthOfLongestSubstring_brute(s: str) -> int:
    max_len = 0
    for i in range(len(s)):
        for j in range(i + 1, len(s) + 1):
            if len(s[i:j]) == len(set(s[i:j])):   # all unique?
                max_len = max(max_len, j - i)
    return max_len

# Test
print(lengthOfLongestSubstring_brute("abcabcbb"))   # 3
print(lengthOfLongestSubstring_brute("bbbbb"))      # 1
print(lengthOfLongestSubstring_brute("pwwkew"))     # 3

## Step-by-Step: Sliding Window

Trace through `"abcabcbb"` with `char_index = {}`, `left = 0`:

| right | char | char_index | In window? | Action | window | max_len |
|-------|------|-----------|-----------|--------|--------|---------|
| 0 | a | {a:0} | No | add | [a] | 1 |
| 1 | b | {a:0,b:1} | No | add | [ab] | 2 |
| 2 | c | {a:0,b:1,c:2} | No | add | [abc] | 3 |
| 3 | a | {a:3,b:1,c:2} | Yes (idx 0 ≥ left 0) | left=1 | [bca] | 3 |
| 4 | b | {a:3,b:4,c:2} | Yes (idx 1 ≥ left 1) | left=2 | [cab] | 3 |
| ... | | | | | | 3 |

When `a` repeats at right=3: left jumps from 0 to 1 (past old a). Window stays valid.

In [ ]:
# Optimal — O(n) time, O(min(n,m)) space
# Sliding window with char→index map for O(1) jumps.

def lengthOfLongestSubstring(s: str) -> int:
    char_index = {}   # char → most recent index
    left = 0
    max_len = 0

    for right, char in enumerate(s):
        # If char is in window (index >= left), jump left past it
        if char in char_index and char_index[char] >= left:
            left = char_index[char] + 1
        char_index[char] = right
        max_len = max(max_len, right - left + 1)

    return max_len

# Test
print(lengthOfLongestSubstring("abcabcbb"))   # 3
print(lengthOfLongestSubstring("bbbbb"))      # 1
print(lengthOfLongestSubstring("pwwkew"))     # 3
print(lengthOfLongestSubstring(""))           # 0
print(lengthOfLongestSubstring("au"))         # 2

## Complexity Analysis

| Approach | Time | Space | Notes |
|----------|------|-------|-------|
| Brute Force | O(n³) | O(min(n,m)) | Build+check every substring |
| Sliding Window (set) | O(n) | O(min(n,m)) | But shrinks one step at a time |
| **Sliding Window + index map** | **O(n)** | **O(min(n,m))** | Jumps left directly |

**Space note:** `m` = size of the character set. For ASCII: 128. For Unicode: up to thousands. In practice, the map never holds more than `min(n, charset_size)` entries.

## Interview Q&A

**Q: Why check `char_index[char] >= left`?**
A: A character might be in `char_index` but its stored index is to the left of the current window. We only care about repeats *within the current window*. If the stored index is before `left`, this character isn't actually in our window — it's safe to ignore.

**Q: What's the difference between using a set vs a dict here?**
A: Set gives O(1) membership testing but you'd need to shrink left one step at a time (O(n) per shrink in worst case = O(n²) total). Dict stores the index, enabling O(1) jumps.

**Q: What's `min(n, m)` in the space complexity?**
A: `n` = string length, `m` = character set size. The map holds at most the smaller of the two — you can't have more unique chars in the window than the entire string has, and you can't exceed the alphabet size.

**Q: What changes for Unicode vs ASCII?**
A: Only the constant in space complexity. The algorithm is the same.

## The Citi Angle

**The "longest run of unique values" pattern:** In telemetry streams, you often need the longest consecutive sequence of distinct server IDs (deduplication run), distinct metric types, or distinct alert codes.

```python
# Longest run of distinct servers in a streaming event log
events = ["srv-01", "srv-02", "srv-01", "srv-03", "srv-04", "srv-02"]
server_idx = {}
left = 0
max_run = 0

for right, server in enumerate(events):
    if server in server_idx and server_idx[server] >= left:
        left = server_idx[server] + 1
    server_idx[server] = right
    max_run = max(max_run, right - left + 1)

print(f"Longest run of distinct servers: {max_run}")   # 3
```

**Interview tie-in:** "The sliding window with a hash map is the pattern for any contiguous-window uniqueness problem. I've applied this to detect the longest clean sequence in a telemetry stream before a repeated alert fired."

## Summary

| | Value |
|--|--|
| **Pattern** | Sliding Window + HashMap (char → last index) |
| **Time** | O(n) |
| **Space** | O(min(n, charset)) |
| **Key line** | `if char in char_index and char_index[char] >= left: left = char_index[char] + 1` |
| **Say in interview** | "HashMap stores last index of each char. Jump left past the previous occurrence in O(1)." |

**The window invariant:** `s[left:right+1]` always contains unique characters.